# Generating a banded, periodic microstructure using SynthetMic

In this notebook we give an example of how to use [SynthetMic](https://github.com/synthetic-microstructures/synthetmic) to generate a 3D, banded, periodic microstructure (Laguerre diagram) with 10,000 grains. The volumes of the grains within each band are drawn from a lognormal distribution. Moreover, we show how this example can be reproduced using our graphical user interface [SynthetMic-GUI](https://david-bourne.shinyapps.io/synthetmic-gui/). 

Note that the SynthetMic library can be installed via pip as follows: `pip install synthetmic`.

First we import the relevant libraries:

In [1]:
import numpy as np
from synthetmic import LaguerreDiagramGenerator
from synthetmic.plot import plot_cells_as_pyvista_fig
import time

Next we define the 3D box $\{ (x,y,z) \,: \, 0 \le x \le L_1, \; 0 \le y \le L_2, \; 0 \le z \le L_3 \}$:

In [2]:
L1 = 2. # length of the box in the x-direction 
L2 = 2. # length of the box in the y-direction
L3 = 2. # length of the box in the z-direction
box = np.array([[0, L1],[0, L2],[0, L3]])

Next we specify the periodicity of the box. We choose it to be periodic in the $x$-direction and non-periodic in the $y$- and $z$-directions.

In [3]:
periodic = [True,False,False]

To generate a banded microstructure, we need to specify the volumes of the grains within each band. We can do this with the following function, which also initialises the positions of the seeds.

In [4]:
def banded_seeds_volumes_3d(N,t,s,box):
    ''' This function generates the grain volumes and seeds for a banded microstructure 
    with len(N) horizontal layers. The volumes of the grains within each layer are drawn from a
    lognormal distribution.
    
    Parameters
    ----------
    N : list of ints
        Number of grains in each layer. The length of N is the number of layers. 
        E.g., N = [1000,8000,1000] corresponds to 3 layers with 1000 grains in the bottom layer, 
        8000 grains in the middle layer, and 1000 grains in the top layer.

    t : 1-D numpy array
        Relative thickness of each layer, e.g., t = np.array([1,0.25,1]) corresponds to 3 layers,
        where the middle layer is 1/4 of the thickness of the top and bottom layers.

    s : 1-D numpy array
        Standard deviation of the volumes of the grains in each layer, relative to the mean layer volume,
        i.e., s[i] = (standard deviation of the grain volumes in layer i)/(mean grain volume in layer i).
        E.g., s = np.array([0.2,0.5,0.2]) corresponds to 3 layers. In the middle layer the ratio of the
        standard deviation to the mean grain volume is 0.5. In the top and bottom layers the ratio of the
        standard deviation to the mean grain volume is 0.2.

    box: 2-D numpy array, shape = (3,2)
        The minimum and maximum coordinates of the box in each dimension.
        E.g., box = np.array([[0, L1],[0, L2],[0, L3]])
        
    Returns
    ----------

    target_vols : 1-D numpy array
        Target volumes of the grains. The length of target_vols in the number of grains.

    seeds : 2-D numpy array, shape = (number of grains,3)
        Coordinates of the seeds. 

    '''

    # height of the box
    L3 = box[2,1]-box[2,0]
    
    # t_hat[i] is the thickness of layer i
    t_hat = L3*t/np.sum(t)

    # mean grain volume in each layer
    L1 = box[0,1]-box[0,0] # length of the box in the x-direction 
    L2 = box[1,1]-box[1,0] # length of the box in the y-direction 
    ln_mean = L1*L2*t_hat/np.array(N) # ln_mean[i] is the mean volume of the grains in layer i

    # standard deviation of the grain volumes in each layer
    std_dev = s*ln_mean

    # lognormal parameters (see here https://en.wikipedia.org/wiki/Log-normal_distribution)
    Sigma = np.sqrt(np.log(1+(std_dev/ln_mean)**2))
    Mu = -0.5*Sigma**2+np.log(ln_mean)

    # Draw the volumes of the grains within each layer from a lognormal distribution 
    vols = []
    for k,n in enumerate(N): # for each layer
        vols.append(np.random.lognormal(Mu[k],Sigma[k],n))
        vols[k] = L1*L2*t_hat[k]*vols[k]/np.sum(vols[k])
        
    target_vols = np.hstack(vols)

    # Distribute the seeds uniformly in each layer
    offset = np.hstack([np.array([0]), np.cumsum(t_hat)])
    x = []
    for k,n in enumerate(N): # for each layer
        x.append(np.random.rand(n,3)@np.diag([L1,L2,t_hat[k]])+np.array([box[0,0],box[1,0],offset[k]]))

    seeds = np.vstack(x)

    return target_vols, seeds

In this example we generate a banded microstructure with 3 layers with 10,00 grains in total, where there are 8000 grains in the middle layer and 1000 grains in the top and bottom layers. We take the middle layer to be 1/4 of the thickness of the top and bottom layers. Within each layer, we take the ratio of the standard derivation to the mean grain volume to be 0.1. 

In [5]:
target_vols, seeds = banded_seeds_volumes_3d([1000,8000,1000],np.array([1,0.25,1]),np.array([0.1,0.1,0.1]),box)

Now we generate the Laguerre diagram using Algorithm 2 from [this paper](https://www.tandfonline.com/doi/full/10.1080/14786435.2020.1790053). The volumes of the grains agree with the target volumes to within the specified tolerance of 1%.

In [6]:
percent_tol = 1. # Volume tolerance: maximum percentage error of the volumes of the grains
num_Lloyd = 5 # Number of Lloyd iterations. The Lloyd iterations regularise the microstructure

# Initialise a LaguerreDiagramGenerator object
diagram = LaguerreDiagramGenerator(tol=percent_tol, n_iter=num_Lloyd) 

# Call the fit method to generate a Laguerre diagram with grains of volumes target_vols (to within the specified tolerance percent_tol)
start = time.time()
diagram.fit(
        seeds=seeds,
        volumes=target_vols,
        domain=box, periodic=periodic
)
end = time.time()
print(f'Run time = {end-start} seconds') 

iteration: 1/5, max_percentage_error: 0.0020%, mean_percentage_error: 0.0000%
Resetting weights to init_weights
iteration: 2/5, max_percentage_error: 0.0332%, mean_percentage_error: 0.0000%
iteration: 3/5, max_percentage_error: 0.0667%, mean_percentage_error: 0.0000%
iteration: 4/5, max_percentage_error: 0.0280%, mean_percentage_error: 0.0000%
iteration: 5/5, max_percentage_error: 0.1325%, mean_percentage_error: 0.0002%
Run time = 14.146441221237183 seconds


We can plot the Laguerre diagram as follows:

In [7]:
pl = plot_cells_as_pyvista_fig(generator=diagram, colorby=diagram.get_fitted_volumes(), notebook=True)
pl.show()

Widget(value='<iframe src="http://localhost:32845/index.html?ui=P_0x7fe445ff2f60_0&reconnect=auto" class="pyvi…

## Reproducing this example using the graphical user interface SynthetMic-GUI

Below we save the target volumes and seeds as csv files. These files can then be used to reproduce this example with our graphical user interface [SynthetMic-GUI](https://david-bourne.shinyapps.io/synthetmic-gui/) as follows: 

* from the drop-down menu **Choose a phase**, select `upload target volumes` and upload the file volumes.csv;
* from the drop-down menu **Choose how seeds are initialized**, select `upload` and upload the file seeds.csv.

You also need to specify the box dimensions and the periodicity in the GUI.

In [8]:
import pandas as pd
df_tv = pd.DataFrame(target_vols,columns=['volumes'])
df_tv.to_csv('volumes.csv',index=False)

df_seeds = pd.DataFrame(seeds,columns=['x','y','z'])
df_seeds.to_csv('seeds.csv',index=False)